# Holdout8 v9 — 실제 축 정렬·주기 필터·제한적 Top-10 폴백

v8의 HCX 측정값을 고정하고 검색·좌표 단계만 다시 실행합니다. 입력 번들은 `holdout8_v9_axis_fallback_colab_input_bundle.zip` 하나입니다.


In [ ]:
# 1. GPU 확인
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=True)
print(gpu.stdout.strip())
assert 'GPU' in gpu.stdout, 'Colab 런타임을 GPU로 변경하세요.'


In [ ]:
# 2. 번들 업로드와 안전한 해제
from google.colab import files
from pathlib import Path
import csv, hashlib, io, json, os, shutil, subprocess, sys, zipfile

os.chdir('/content')
uploaded = files.upload()
bundles = [name for name in uploaded if name.endswith('.zip')]
assert len(bundles) == 1, 'v9 입력 번들 ZIP 하나만 업로드하세요.'
ROOT = Path('/content/holdout8_gpu_v9')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(uploaded[bundles[0]])) as archive:
    for info in archive.infolist():
        parts = [p for p in info.filename.replace('\\', '/').split('/') if p not in {'', '.'}]
        assert '..' not in parts, f'안전하지 않은 ZIP 경로: {info.filename}'
        target = ROOT.joinpath(*parts)
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
assert (ROOT / 'data/holdout8_stratified_articles.csv').is_file(), '번들 구조가 올바르지 않습니다.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'outputs/holdout8_stratified48_v9'
MAP = OUT / '07_mapping_v2'
SEM = ROOT / 'data/indexes/kosis_bge_m3'
CHR = ROOT / 'data/indexes/kosis_meta_chroma_holdout8_v9'
OUT.mkdir(parents=True, exist_ok=True)

def run(args):
    cmd = [str(x) for x in args]
    print('\n$', ' '.join(cmd[:3]), '...', flush=True)
    process = subprocess.Popen(cmd, cwd=ROOT, env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, cmd)
    return subprocess.CompletedProcess(cmd, returncode)

def csv_rows(path):
    with open(path, encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

print('ROOT =', ROOT)


In [ ]:
# 3. 의존성 설치
%pip install -q -r requirements.txt -r requirements-ml.txt
import chromadb, pandas as pd, sentence_transformers, torch
assert torch.cuda.is_available(), 'CUDA를 사용할 수 없습니다.'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))


In [ ]:
# 4. KOSIS API 키
from getpass import getpass
try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret(name):
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            pass
    return value or getpass(f'{name}: ')

os.environ['KOSIS_API_KEY'] = secret('KOSIS_API_KEY')
assert os.environ['KOSIS_API_KEY']
print('KOSIS_API_KEY가 메모리에 설정됐습니다.')


In [ ]:
# 5. 번들 무결성 검증과 고정 체크포인트 복원
manifest = json.loads((ROOT / 'bundle_manifest.json').read_text(encoding='utf-8'))
for rel, expected in manifest['files'].items():
    actual = hashlib.sha256((ROOT / rel).read_bytes()).hexdigest()
    assert actual == expected, f'해시 불일치: {rel}'
for name in ['01_sentences.csv', '03_claim_contexts.csv', '05_hcx_measurements.csv']:
    shutil.copy2(ROOT / 'data/checkpoints' / name, OUT / name)
articles = csv_rows(ROOT / 'data/holdout8_stratified_articles.csv')
measurements = csv_rows(OUT / '05_hcx_measurements.csv')
assert len(articles) == 48 and len(measurements) == 769
print('고정 입력:', len(articles), 'articles /', len(measurements), 'measurements')


In [ ]:
# 6. KOSIS 표 BGE-M3 인덱스
run([sys.executable, 'kosis_build_embedding_index.py', '--table-index', 'data/reference/kosis_table_summary.csv', '--out-dir', SEM, '--batch-size', '16', '--device', 'cuda'])
assert (SEM / 'manifest.json').is_file()


In [ ]:
# 7. 고정 measurement에서 표 후보와 공식 메타 생성
run([sys.executable, 'run_kosis_measurement_pipeline.py', '--input', OUT / '05_hcx_measurements.csv', '--table-index', ROOT / 'data/reference/kosis_table_summary.csv', '--semantic-index', SEM, '--out-dir', MAP, '--retrieval-mode', 'hybrid', '--semantic-top-k', '50', '--rerank-top-k', '60', '--top-tables', '20', '--lexical-reserve-k', '10', '--top-rank-for-meta', '20', '--device', 'cuda'])
READY = MAP / '05_hcx_measurements_kosis_ready.csv'
META = MAP / '05_hcx_measurements_kosis_meta_index.csv'
TABLE_CAND = MAP / '05_hcx_measurements_kosis_table_candidates.csv'
EVAL = MAP / 'evaluation_set.csv'
run([sys.executable, 'lock_evaluation_set.py', '--ready', READY, '--output', EVAL, '--excluded-output', MAP / 'evaluation_set_excluded.csv', '--manifest', MAP / 'evaluation_set_manifest.json'])
print('evaluation rows =', len(csv_rows(EVAL)))


In [ ]:
# 8. 실제 축 이름을 포함한 Chroma v2 좌표 인덱스
run([sys.executable, 'kosis_build_chroma_meta_index.py', '--meta-index', META, '--persist-dir', CHR, '--collection', 'kosis_meta_coordinates', '--embedding-model', 'BAAI/bge-m3', '--axis-value-limit', '300', '--prd-se-source', TABLE_CAND, '--device', 'cuda', '--reset'])
assert (CHR / 'chroma_manifest.json').is_file()
print((CHR / 'chroma_manifest.json').read_text(encoding='utf-8')[:1500])


In [ ]:
# 9. 주기 사전 필터가 적용된 Top-5/Top-10 검색과 실제 축 우선 선택
EVAL_V9 = MAP / 'evaluation_set_v9_enriched.csv'
run([sys.executable, 'enrich_mcp_gold_200_inputs.py', '--input', EVAL, '--output', EVAL_V9, '--stats', MAP / 'v9_enrichment_stats.json'])
CAND5 = MAP / 'chroma_candidates_top5.csv'
CAND10 = MAP / 'chroma_candidates_top10.csv'
for table_k, output, stats in [('5', CAND5, MAP / 'chroma_stats_top5.csv'), ('10', CAND10, MAP / 'chroma_stats_top10.csv')]:
    run([sys.executable, 'kosis_chroma_hybrid_search.py', '--claims', EVAL_V9, '--table-candidates', TABLE_CAND, '--persist-dir', CHR, '--collection', 'kosis_meta_coordinates', '--output', output, '--stats-output', stats, '--table-top-k', table_k, '--dense-top-k', '50', '--lexical-top-k', '50', '--rerank-top-k', '60', '--final-top-k', '30', '--min-candidates-per-table', '3', '--reranker-model', 'BAAI/bge-reranker-v2-m3', '--device', 'cuda'])
SELECT5 = MAP / 'two_stage_selected_top5.csv'
SELECT10 = MAP / 'two_stage_selected_top10.csv'
for candidates, selected in [(CAND5, SELECT5), (CAND10, SELECT10)]:
    run([sys.executable, 'select_mcp_gold_200_two_stage_coordinates.py', '--claims', EVAL_V9, '--candidates', candidates, '--output', selected, '--item-top-k', '10'])
print('selected =', len(csv_rows(SELECT5)), len(csv_rows(SELECT10)))


In [ ]:
# 10. Top-5 검증, 빈 응답 OBJ 완화, 실패 건만 Top-10 폴백
VALID5 = MAP / 'chroma_validated_top5.csv'
FALLBACK_IN = MAP / 'top10_fallback_input.csv'
FALLBACK_VALID = MAP / 'top10_fallback_validated.csv'
FINAL = MAP / 'chroma_validated_bounded_fallback.csv'

def validate_selected(selected, output):
    run([sys.executable, 'kosis_validate_mapping_candidates.py', '--input', selected, '--meta-index', META, '--output', output, '--evaluate-all-ranks', '--strict-seeded-coordinate', '--item-top-k', '1', '--obj-top-k', '1', '--max-combinations', '1', '--allow-provisional', '--relax-empty-obj', '--max-relaxed-requests', '4', '--max-relaxed-obj-drops', '2'])

validate_selected(SELECT5, VALID5)
run([sys.executable, 'kosis_topk_fallback.py', 'prepare', '--primary-validated', VALID5, '--fallback-candidates', SELECT10, '--output', FALLBACK_IN])
if csv_rows(FALLBACK_IN):
    validate_selected(FALLBACK_IN, FALLBACK_VALID)
else:
    FALLBACK_VALID.write_text('', encoding='utf-8')
run([sys.executable, 'kosis_topk_fallback.py', 'merge', '--primary-validated', VALID5, '--fallback-validated', FALLBACK_VALID, '--output', FINAL])
final_rows = csv_rows(FINAL)
assert len(final_rows) == len(csv_rows(VALID5))
assert all(r.get('mapping_status') != 'READY' for r in final_rows if r.get('obj_relaxation_used') == 'Y')
print('Top-10 fallback input rows =', len(csv_rows(FALLBACK_IN)))


In [ ]:
# 11. MCP 실제 좌표 골드 Top-k 평가와 주기 음성 골드 확인
ACTUAL_EVAL = MAP / 'actual_coordinate_eval'
run([sys.executable, 'evaluate_mcp_gold_200_mapping.py', '--gold', ROOT / 'data/gold/holdout8_v9_mcp_coordinate_gold.csv', '--candidates', CAND10, '--mapped', FINAL, '--ks', '1', '5', '10', '--output-dir', ACTUAL_EVAL])
coord_summary = json.loads((ACTUAL_EVAL / 'summary.json').read_text(encoding='utf-8'))
negative = csv_rows(ROOT / 'data/gold/holdout8_v9_period_negative_gold.csv')
candidates = csv_rows(CAND10)
def key(row):
    return row.get('claim_measurement_id') or row.get('claim_id') or ''
violations = []
for gold in negative:
    violations.extend(row for row in candidates if key(row) == gold['claim_measurement_id'] and row.get('org_id') == gold['org_id'] and row.get('tbl_id') == gold['tbl_id'])
period_metrics = {'gold_rows': len(negative), 'violations': len(violations), 'filter_accuracy': (len(negative) - len({key(r) for r in violations})) / len(negative)}
(MAP / 'period_negative_metrics.json').write_text(json.dumps(period_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
assert not violations, '주기 불일치 표가 검색 후보에 남았습니다.'
print(json.dumps({'coordinate': coord_summary, 'period_negative': period_metrics}, ensure_ascii=False, indent=2))


In [ ]:
# 12. 최종 READY 실제값 검증
v = pd.read_csv(FINAL, dtype=str, keep_default_na=False)
ready_v = v[v['mapping_status'].eq('READY')].copy()
ready_v['_rank'] = pd.to_numeric(ready_v.get('candidate_rank', '999'), errors='coerce').fillna(999)
ready_v = ready_v.sort_values('_rank').drop_duplicates('claim_measurement_id').drop(columns='_rank')
VERIFY_IN = MAP / 'verify_input.csv'
VERIFIED = MAP / 'verified.csv'
ready_v.to_csv(VERIFY_IN, index=False, encoding='utf-8-sig')
if len(ready_v):
    run([sys.executable, 'kosis_verify_claim_values.py', '--input', VERIFY_IN, '--output', VERIFIED, '--delay', '0.12'])
    vf = pd.read_csv(VERIFIED, dtype=str, keep_default_na=False)
    assert not vf['verdict_code'].eq('KOSIS_API_ERROR').any()
else:
    vf = pd.DataFrame()
print('verified READY rows =', len(vf))


In [ ]:
# 13. 요약과 결과 ZIP 다운로드
from collections import Counter
final_rows = csv_rows(FINAL)
summary = {
    'articles': len(articles), 'measurements': len(measurements),
    'evaluation_measurements': len(csv_rows(EVAL)),
    'validated_status_rows': dict(Counter(r.get('mapping_status', '') for r in final_rows)),
    'obj_relaxed_rows': sum(r.get('obj_relaxation_used') == 'Y' for r in final_rows),
    'top10_fallback_attempted': sum(r.get('topk_fallback_attempted') == 'Y' for r in final_rows),
    'top10_fallback_recovered': sum(r.get('topk_fallback_recovered') == 'Y' for r in final_rows),
    'actual_coordinate_metrics': coord_summary, 'period_negative_metrics': period_metrics,
    'api_error_rows': sum(r.get('mapping_status') == 'API_ERROR' for r in final_rows),
}
(MAP / 'gpu_run_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
for src, name in [(SEM / 'manifest.json', 'semantic_manifest.json'), (CHR / 'chroma_manifest.json', 'chroma_manifest.json')]:
    shutil.copy2(src, MAP / name)
archive = shutil.make_archive('/content/holdout8_v9_gpu_results', 'zip', OUT)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('다운로드:', archive)
files.download(archive)


## 완료

다운로드한 `holdout8_v9_gpu_results.zip`을 Codex 작업에 첨부하세요. 대용량 Chroma 인덱스는 결과 ZIP에 포함하지 않습니다.
